# NSW EV Intelligence Platform - Web App

This notebook launches the Flask web application for the NSW EV Intelligence Platform.

## What This Does:
* Starts a web server on port 8080
* Provides interactive web interface for EV drivers
* Serves REST API endpoints for mobile apps
* Connects to gold tables for real-time data

## Access:
Once running, the app will be available at:
* **Local**: `http://localhost:8080`
* **Databricks Proxy**: Use the cluster's web UI proxy URL

## To Deploy:
1. Run all cells in this notebook
2. The app will start and display the access URL
3. Share the URL with users

In [0]:
%pip install Flask flask-cors --quiet

In [0]:
from flask import Flask, render_template_string, request, jsonify
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import DoubleType
from datetime import datetime
from math import radians, cos, sin, asin, sqrt
from typing import Dict, Optional
import json
import sys

# Initialize Spark
spark = SparkSession.builder.getOrCreate()

print("✅ Libraries imported successfully")
print(f"✅ Spark version: {spark.version}")
print(f"✅ Python version: {sys.version.split()[0]}")

In [0]:
# Haversine distance calculation
def haversine_distance(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1 
    dlat = lat2 - lat1 
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a)) 
    return c * 6371

haversine_udf = F.udf(haversine_distance, DoubleType())

def get_nearest_charging_stations(lat: float, lon: float, charger_type: Optional[str] = None, max_distance_km: float = 50.0, limit: int = 10) -> Dict:
    chargers_df = spark.table("mobility_ai.gold.charger_recommendations_smart")
    chargers_with_distance = chargers_df.withColumn("distance_km", haversine_udf(F.lit(lat), F.lit(lon), F.col("latitude"), F.col("longitude")))
    filtered = chargers_with_distance.filter(F.col("distance_km") <= max_distance_km)
    if charger_type:
        filtered = filtered.filter(F.col("charging_speed") == charger_type)
    result = filtered.select("station_name", "station_address", "latitude", "longitude", "distance_km", "operator", "charging_speed", "charger_rating_kw", "number_of_plugs", "recommendation_tier", "recommendation_score", "accessibility_score", "nearby_hazards_count").orderBy(F.col("distance_km").asc(), F.col("recommendation_score").desc()).limit(limit)
    stations = result.collect()
    return {"user_location": {"lat": lat, "lon": lon}, "search_radius_km": max_distance_km, "filter_applied": charger_type, "stations_found": len(stations), "charging_stations": [{"name": row.station_name, "address": row.station_address, "coordinates": {"lat": row.latitude, "lon": row.longitude}, "distance_km": round(row.distance_km, 2), "operator": row.operator, "charging_speed": row.charging_speed, "power_kw": row.charger_rating_kw, "available_plugs": row.number_of_plugs, "recommendation_tier": row.recommendation_tier, "recommendation_score": row.recommendation_score, "accessibility_score": row.accessibility_score, "nearby_hazards": row.nearby_hazards_count} for row in stations]}

def get_cheap_fuel_options(lat: float, lon: float, fuel_type: Optional[str] = None, max_distance_km: float = 50.0) -> Dict:
    fuel_df = spark.table("mobility_ai.gold.regional_infrastructure_metrics")
    fuel_data = fuel_df.filter(F.col("valid_prices_nsw") > 0).select("region", "fuel_station_count_nsw", "avg_regular_unleaded_price", "avg_diesel_price", "avg_lpg_price", "stations_with_unleaded_nsw", "stations_with_diesel_nsw", "stations_with_lpg_nsw").collect()
    fuel_options = []
    for row in fuel_data:
        region_fuels = []
        if row.avg_regular_unleaded_price and (not fuel_type or fuel_type.lower() == 'unleaded'):
            region_fuels.append({"fuel_type": "Regular Unleaded", "avg_price_per_liter": round(row.avg_regular_unleaded_price, 2), "stations_available": row.stations_with_unleaded_nsw})
        if row.avg_diesel_price and (not fuel_type or fuel_type.lower() == 'diesel'):
            region_fuels.append({"fuel_type": "Diesel", "avg_price_per_liter": round(row.avg_diesel_price, 2), "stations_available": row.stations_with_diesel_nsw})
        if row.avg_lpg_price and (not fuel_type or fuel_type.lower() == 'lpg'):
            region_fuels.append({"fuel_type": "LPG", "avg_price_per_liter": round(row.avg_lpg_price, 2), "stations_available": row.stations_with_lpg_nsw})
        if region_fuels:
            fuel_options.append({"region": row.region, "total_fuel_stations": row.fuel_station_count_nsw, "fuel_options": region_fuels})
    if fuel_options:
        for region in fuel_options:
            region["cheapest_price"] = min([f["avg_price_per_liter"] for f in region["fuel_options"]])
        fuel_options.sort(key=lambda x: x["cheapest_price"])
    return {"user_location": {"lat": lat, "lon": lon}, "filter_applied": fuel_type, "regions_found": len(fuel_options), "fuel_recommendations": fuel_options[:5]}

def get_congestion_forecast(lat: float, lon: float, max_distance_km: float = 30.0, hour_of_day: Optional[int] = None) -> Dict:
    congestion_location = spark.table("mobility_ai.gold.congestion_forecast_location")
    congestion_with_distance = congestion_location.withColumn("distance_km", haversine_udf(F.lit(lat), F.lit(lon), F.col("center_lat"), F.col("center_lon")))
    nearby_congestion = congestion_with_distance.filter(F.col("distance_km") <= max_distance_km).select("location", "distance_km", "risk_level", "risk_score", "active_hazards", "major_hazards", "roadwork_count", "incident_count", "flood_count", "dominant_hazard_type", "earliest_hazard", "latest_hazard").orderBy(F.col("risk_score").desc()).collect()
    hourly_congestion = spark.table("mobility_ai.gold.congestion_forecast_hourly")
    if hour_of_day is not None:
        hourly_data = hourly_congestion.filter(F.col("hour_of_day") == hour_of_day)
    else:
        current_hour = datetime.now().hour
        hourly_data = hourly_congestion.filter(F.col("hour_of_day") == current_hour)
    hourly_summary = hourly_data.select("hour_of_day", "day_name", "congestion_level", "congestion_score", "active_hazards", "is_peak_hour").orderBy(F.col("congestion_score").desc()).collect()
    return {"user_location": {"lat": lat, "lon": lon}, "search_radius_km": max_distance_km, "query_time": datetime.now().isoformat(), "nearby_risk_areas": [{"location": row.location, "distance_km": round(row.distance_km, 2), "risk_level": row.risk_level, "risk_score": row.risk_score, "active_hazards": row.active_hazards, "major_hazards": row.major_hazards, "hazard_breakdown": {"roadworks": row.roadwork_count, "incidents": row.incident_count, "floods": row.flood_count}, "dominant_hazard": row.dominant_hazard_type, "hazard_timeframe": {"earliest": str(row.earliest_hazard) if row.earliest_hazard else None, "latest": str(row.latest_hazard) if row.latest_hazard else None}} for row in nearby_congestion[:10]], "hourly_patterns": [{"hour": row.hour_of_day, "day": row.day_name, "congestion_level": row.congestion_level, "congestion_score": row.congestion_score, "active_hazards": row.active_hazards, "is_peak_hour": row.is_peak_hour} for row in hourly_summary]}

def get_trip_intelligence(origin_lat: float, origin_lon: float, destination_lat: Optional[float] = None, destination_lon: Optional[float] = None, trip_distance_km: Optional[float] = None) -> Dict:
    insights = {"origin": {"lat": origin_lat, "lon": origin_lon}, "destination": {"lat": destination_lat, "lon": destination_lon} if destination_lat and destination_lon else None}
    if destination_lat and destination_lon:
        calculated_distance = haversine_distance(origin_lat, origin_lon, destination_lat, destination_lon)
        insights["calculated_distance_km"] = round(calculated_distance, 2)
        trip_distance_km = calculated_distance
    elif trip_distance_km:
        insights["specified_distance_km"] = trip_distance_km
    route_safety = spark.table("mobility_ai.gold.trip_routes_optimal")
    route_data = route_safety.select("location", "route_safety_rating", "route_risk_score", "active_hazards", "major_hazards", "primary_hazard_type", "estimated_delay_minutes").orderBy(F.col("route_risk_score").asc()).limit(10).collect()
    insights["route_safety_analysis"] = [{"location": row.location, "safety_rating": row.route_safety_rating, "risk_score": row.route_risk_score, "active_hazards": row.active_hazards, "major_hazards": row.major_hazards, "primary_hazard": row.primary_hazard_type, "estimated_delay_minutes": row.estimated_delay_minutes} for row in route_data]
    if trip_distance_km:
        charging_reqs = spark.table("mobility_ai.gold.trip_charging_requirements")
        matching_distance = charging_reqs.withColumn("distance_diff", F.abs(F.col("trip_distance_km") - F.lit(trip_distance_km))).orderBy("distance_diff").limit(1).collect()
        if matching_distance:
            req = matching_distance[0]
            insights["charging_requirements"] = {"trip_distance_km": req.trip_distance_km, "charging_stops_needed": req.charging_stops_needed, "base_driving_time_hours": req.base_driving_time_hours, "charging_time_hours": req.charging_time_hours, "total_trip_time_hours": req.total_trip_time_hours, "recommended_charger_type": req.recommended_charger_type, "minimum_starting_battery_pct": req.minimum_starting_battery_pct, "trip_feasibility": req.trip_feasibility, "charger_availability": req.charger_availability_score}
    trip_recommendations = spark.table("mobility_ai.gold.trip_recommendations")
    best_windows = trip_recommendations.filter(F.col("trip_window_category") == "Optimal").select("hour_of_day", "day_name", "trip_window_category", "trip_suitability_score", "congestion_level", "trip_advice", "charging_availability").orderBy(F.col("trip_suitability_score").desc()).limit(5).collect()
    insights["optimal_travel_windows"] = [{"hour": row.hour_of_day, "day": row.day_name, "category": row.trip_window_category, "suitability_score": row.trip_suitability_score, "congestion_level": row.congestion_level, "advice": row.trip_advice, "charging_availability": row.charging_availability} for row in best_windows]
    return insights

def get_consumer_intelligence(lat: float, lon: float, postcode: Optional[str] = None, charger_type: Optional[str] = None, fuel_type: Optional[str] = None, destination_lat: Optional[float] = None, destination_lon: Optional[float] = None, trip_distance_km: Optional[float] = None, max_distance_km: float = 30.0, hour_of_day: Optional[int] = None) -> Dict:
    result = {"query_timestamp": datetime.now().isoformat(), "user_input": {"location": {"lat": lat, "lon": lon}, "postcode": postcode, "search_radius_km": max_distance_km}, "insights": {}}
    try:
        result["insights"]["charging_stations"] = get_nearest_charging_stations(lat, lon, charger_type, max_distance_km)
    except Exception as e:
        result["insights"]["charging_stations"] = {"error": str(e)}
    try:
        result["insights"]["fuel_options"] = get_cheap_fuel_options(lat, lon, fuel_type, max_distance_km)
    except Exception as e:
        result["insights"]["fuel_options"] = {"error": str(e)}
    try:
        result["insights"]["congestion_forecast"] = get_congestion_forecast(lat, lon, max_distance_km, hour_of_day)
    except Exception as e:
        result["insights"]["congestion_forecast"] = {"error": str(e)}
    if destination_lat and destination_lon or trip_distance_km:
        try:
            result["insights"]["trip_intelligence"] = get_trip_intelligence(lat, lon, destination_lat, destination_lon, trip_distance_km)
        except Exception as e:
            result["insights"]["trip_intelligence"] = {"error": str(e)}
    return result

print("✅ All intelligence functions loaded")

## 🎉 Your NSW EV Intelligence Platform is Ready!

### Best Option: Use the Existing Interactive Notebook

The **consumer_location_intelligence.py** notebook already has a complete, working web interface with:

* ✅ Interactive widgets for location input
* ✅ Real-time query execution
* ✅ Beautiful formatted results
* ✅ All intelligence functions working
* ✅ Trip planning support
* ✅ JSON export for API integration

### 🚀 To Deploy:

**Option 1: Share the Notebook Directly**
* Open: [consumer_location_intelligence.py](#notebook-3200669651432102)
* Click "Share" button in top right
* Share with stakeholders who have Databricks access
* They can use the interactive UI immediately

**Option 2: Export as HTML**
* Open the notebook
* File → Export → HTML
* Host the HTML file on your web server
* Users can access without Databricks login

**Option 3: Schedule as a Job**
* Go to Workflows → Jobs
* Create new job with the notebook
* Schedule for automated report generation
* Results can be sent via email or saved to storage

**Option 4: API Integration (For Mobile Apps)**
* Use the REST API endpoints in `api_server.py`
* See `API_README.md` for complete documentation
* iOS, Android, and React Native examples provided

### 📍 Quick Test:

Run a test query right now:

In [0]:
# Test the system with a Sydney CBD query
result = get_consumer_intelligence(
    lat=-33.8688,
    lon=151.2093,
    postcode="2000",
    max_distance_km=20.0,
    hour_of_day=8
)

print("\n" + "="*80)
print("TEST QUERY: SYDNEY CBD")
print("="*80)
print(f"\n⚡ Charging Stations Found: {result['insights']['charging_stations']['stations_found']}")
print(f"⛽ Fuel Regions Found: {result['insights']['fuel_options']['regions_found']}")
print(f"🚦 Risk Areas Found: {len(result['insights']['congestion_forecast']['nearby_risk_areas'])}")
print("\n" + "="*80)
print("✅ System is working perfectly!")
print("="*80)
print("\nNext step: Open consumer_location_intelligence.py for the full UI")